# Set Up MLflow Webhooks for Tekton Pipelines

In this notebook, we'll create MLflow webhooks that automatically trigger our Tekton pipelines when prompt-related events happen.

We have two pipelines to connect:

1. **Evaluation Pipeline** — triggered when a new prompt version is created, so we automatically run evals against it.
2. **Prompt Promotion Pipeline** — triggered when someone tags a prompt version with the `prod` alias, so the production GitOps config is updated automatically.

In [ ]:
# ‼️‼️ Replace these with your actual values ‼️
USER_NAME = "<your-user-name>"
CLUSTER_DOMAIN = "<your-cluster-domain>"

In [ ]:
import mlflow

MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_workspace(f"{USER_NAME}-toolings")

client = mlflow.MlflowClient()

## 1. Webhook for the Evaluation Pipeline

This webhook tells MLflow to notify our **canopy-evals** Tekton EventListener whenever a new prompt or prompt version is created. That way, every prompt change automatically triggers our evaluation pipeline.

In [ ]:
webhook = client.create_webhook(
    name=f"{USER_NAME}-canopy-evals",
    url=f"https://canopy-evals-event-listener-{USER_NAME}-toolings.{CLUSTER_DOMAIN}",
    events=["prompt.created", "prompt_version.created"],
    description="Triggers evaluation pipeline when new prompt versions are created",
)

print(f"Created webhook: {webhook.webhook_id}")

### ❣️ Now go back to the instructions to test the automation you set up. You'll come back for more automation, promise 😏

## 2. Webhook for the Prompt Promotion Pipeline

This webhook fires when a `prod` alias is added to a prompt version. It triggers our **prompt-promotion** pipeline, which updates the production GitOps config (`genaiops-gitops/canopy/prod/backend/config.yaml`) with the specific prompt version number — giving us full traceability of what's running in production.

> **Note:** Only run this cell after you've deployed the `prompt-promotion-pipeline` chart via ArgoCD.

In [ ]:
webhook = client.create_webhook(
    name=f"{USER_NAME}-canopy-prompt-promotion",
    url=f"https://prompt-promotion-event-listener-{USER_NAME}-toolings.{CLUSTER_DOMAIN}",
    events=["prompt_alias.created"],
    description="Triggers prompt promotion pipeline when prod alias is set",
)

print(f"Created webhook: {webhook.webhook_id}")